# NISAR Covariance and Speckle Filtering Tutorial

This notebook demonstrates an end-to-end workflow:

1. Read a NISAR polarimetric cube in CYX format
2. Build the covariance matrix with `CovarianceMatrix`
3. Apply third-party speckle filtering (MuLoG and optionally NL-SAR)
4. Build RGB display images from covariance terms
5. Compare pre/post filtering in GRDK dual pane viewer

RGB mapping requested:
- R = XX
- G = XY
- B = YY

In [6]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

# Ensure local source checkouts are importable in the notebook kernel without
# machine-specific absolute paths.
def _add_repo_to_syspath(repo_name, package_dir_name):
    cwd = Path.cwd().resolve()
    for base in [cwd] + list(cwd.parents):
        candidate = base / repo_name
        if (candidate / package_dir_name).exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return candidate
    return None

found_grdl = _add_repo_to_syspath('grdl', 'grdl')
found_grdk = _add_repo_to_syspath('grdk', 'grdk')
found_sarf = _add_repo_to_syspath('grdl-sar-filtering', 'grdl_sar_filtering')

# If another grdl distribution was already imported, drop it and re-import from local path.
for mod_name in list(sys.modules):
    if mod_name == 'grdl' or mod_name.startswith('grdl.'):
        del sys.modules[mod_name]

import grdl
print('Using grdl from:', grdl.__file__)
print('Discovered repos:', {'grdl': found_grdl, 'grdk': found_grdk, 'grdl-sar-filtering': found_sarf})

from grdl.IO.sar.nisar import NISARReader
from grdl.IO.models.base import ImageMetadata, ChannelMetadata
from grdl.image_processing.decomposition.pol_matrix import CovarianceMatrix
from grdl_sar_filtering import create_speckle_filter
from grdk.viewers.dual_viewer import DualGeoViewer

print('Imports OK')

Using grdl from: /home/jason/clones/grdx/grdl/grdl/__init__.py
Discovered repos: {'grdl': PosixPath('/home/jason/clones/grdx/grdl'), 'grdk': PosixPath('/home/jason/clones/grdx/grdk'), 'grdl-sar-filtering': PosixPath('/home/jason/clones/grdx/grdl-sar-filtering')}
Imports OK


## Configuration

Set `nisar_file` to a real NISAR RSLC file if available. If not, the next cell creates synthetic dual-pol data.

In [14]:
import logging

# Enable INFO logs so pol_matrix emits channel extraction and assignment details.
logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] %(name)s: %(message)s',
    force=True,
)

# Focus on pol_matrix logs specifically.
# logging.getLogger('grdl.image_processing.decomposition.pol_matrix').setLevel(logging.INFO)
logging.getLogger('grdl').setLevel(logging.INFO)

# print('INFO logging enabled for grdl.image_processing.decomposition.pol_matrix')
print('INFO logging enabled for grdl')
# print('Input channel metadata order:')
# for i, cm in enumerate(getattr(metadata, 'channel_metadata', []) or []):
#     print(
#         f"  idx={i} name={getattr(cm, 'name', None)} pol={getattr(cm, 'polarization', None)} role={getattr(cm, 'role', None)}"
#     )

# print('Run the covariance cell next to see resolved co-pol/cross-pol mapping logs.')

INFO logging enabled for grdl


In [15]:
nisar_file = Path(
    '/mnt/c/Projects/SAR/data/NISAR/20260105T055924_20260105T055931/NISAR_L1_PR_RSLC_009_116_D_054_2005_QPDH_A_20260105T055924_20260105T055931_X05010_N_P_J_001.h5')
# None  # Example: Path('/path/to/NISAR_L1_RSLC_*.h5')

frequency = 'A'
polarizations = 'all'

chip_size = 256
cov_window_size = 9
number_looks = 3.0

run_mulog = True
run_nlsar = False  # Set True only if nlsartoolbox is available

print(f'chip_size={chip_size}, cov_window_size={cov_window_size}, looks={number_looks}')

chip_size=256, cov_window_size=9, looks=3.0


In [16]:
def _synthetic_dual_pol_cube(rows=256, cols=256, seed=7):
    rng = np.random.default_rng(seed)

    amp_xx = rng.exponential(scale=1.0, size=(rows, cols)).astype(np.float32)
    amp_xy = rng.exponential(scale=0.6, size=(rows, cols)).astype(np.float32)
    amp_yy = rng.exponential(scale=1.0, size=(rows, cols)).astype(np.float32)

    ph_xx = rng.uniform(0.0, 2.0 * np.pi, size=(rows, cols))
    ph_xy = rng.uniform(0.0, 2.0 * np.pi, size=(rows, cols))
    ph_yy = rng.uniform(0.0, 2.0 * np.pi, size=(rows, cols))

    sxx = (amp_xx * np.exp(1j * ph_xx)).astype(np.complex64)
    sxy = (amp_xy * np.exp(1j * ph_xy)).astype(np.complex64)
    syy = (amp_yy * np.exp(1j * ph_yy)).astype(np.complex64)

    cube = np.stack([sxx, sxy, syy], axis=0)

    metadata = ImageMetadata(
        format='SYNTHETIC',
        rows=rows,
        cols=cols,
        bands=3,
        dtype='complex64',
        axis_order='CYX',
        channel_metadata=[
            ChannelMetadata(index=0, name='XX', polarization='XX', role='co_pol_a'),
            ChannelMetadata(index=1, name='XY', polarization='XY', role='cross_pol'),
            ChannelMetadata(index=2, name='YY', polarization='YY', role='co_pol_b'),
        ],
    )
    return cube, metadata

if nisar_file is not None and Path(nisar_file).exists():
    reader = NISARReader(nisar_file, frequency=frequency, polarizations=polarizations)
    metadata = reader.metadata

    if chip_size is None:
        cube = reader.read_full()
    else:
        r0 = max(0, (metadata.rows - chip_size) // 2)
        c0 = max(0, (metadata.cols - chip_size) // 2)
        r1 = min(metadata.rows, r0 + chip_size)
        c1 = min(metadata.cols, c0 + chip_size)
        cube = reader.read_chip(r0, r1, c0, c1)
    reader.close()
else:
    cube, metadata = _synthetic_dual_pol_cube(rows=chip_size or 256, cols=chip_size or 256)

print('Cube shape:', cube.shape, 'dtype:', cube.dtype)
print('Axis order:', metadata.axis_order)
print('Channels:', [cm.polarization for cm in metadata.channel_metadata])

[INFO] grdl.IO.sar.nisar: Loaded NISAR RSLC NISAR_L1_PR_RSLC_009_116_D_054_2005_QPDH_A_20260105T055924_20260105T055931_X05010_N_P_J_001.h5 (12160 x 26124) freq=A pol=HV,VV,VH,HH


Cube shape: (4, 256, 256) dtype: complex64
Axis order: CYX
Channels: ['HV', 'VV', 'VH', 'HH']


## Covariance Matrix

`CovarianceMatrix.execute` returns **CCYX**: `(N, N, rows, cols)`.
The filters expect **YXCC**: `(rows, cols, N, N)`.

In [ ]:
cmat = CovarianceMatrix(window_size=cov_window_size)
cov_ccyx, cov_meta = cmat.execute(metadata, cube)
cov_yxcc = np.transpose(cov_ccyx, (2, 3, 0, 1))

print('cov_ccyx shape:', cov_ccyx.shape)
print('cov_yxcc shape:', cov_yxcc.shape)

In [ ]:
def cov_to_rgb(cov):
    """Map covariance to RGB with R=XX, G=XY, B=YY."""
    # cov shape: (rows, cols, N, N)
    xx = np.real(cov[..., 0, 0])
    xy = np.real(cov[..., 0, 1])
    yy = np.real(cov[..., -1, -1])

    rgb = np.stack([xx, xy, yy], axis=-1).astype(np.float32)

    # SAR-friendly display scaling
    rgb = np.log10(np.maximum(rgb, 1e-8))

    out = np.empty_like(rgb, dtype=np.float32)
    for b in range(3):
        lo, hi = np.percentile(rgb[..., b], [2.0, 98.0])
        out[..., b] = np.clip((rgb[..., b] - lo) / max(hi - lo, 1e-8), 0.0, 1.0)
    return out

rgb_pre = cov_to_rgb(cov_yxcc)

plt.figure(figsize=(6, 6))
plt.imshow(rgb_pre)
plt.title('Pre-filter covariance RGB (R=XX, G=XY, B=YY)')
plt.axis('off')
plt.show()

## Speckle Filtering

Apply MuLoG and optionally NL-SAR on the YXCC covariance cube.

In [ ]:
results = {'pre': cov_yxcc}

if run_mulog:
    mulog = create_speckle_filter('mulog', number_looks=number_looks)
    results['mulog'] = mulog.apply(cov_yxcc)
    print('MuLoG complete:', results['mulog'].shape)

if run_nlsar:
    nlsar = create_speckle_filter('nlsar', number_looks=number_looks)

    # Build a homogeneous noise covariance image for NL-SAR statistics.
    m, n, d, _ = cov_yxcc.shape
    mean_power = float(np.median(np.real(np.trace(cov_yxcc, axis1=-2, axis2=-1))))
    noise_image = np.zeros((m, n, d, d), dtype=np.complex64)
    for i in range(d):
        noise_image[..., i, i] = mean_power / max(d, 1)

    results['nlsar'] = nlsar.apply(cov_yxcc, noise_image=noise_image)
    print('NL-SAR complete:', results['nlsar'].shape)

In [ ]:
rgb_results = {k: cov_to_rgb(v) for k, v in results.items()}

ncols = len(rgb_results)
fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6))
if ncols == 1:
    axes = [axes]

for ax, (name, img) in zip(axes, rgb_results.items()):
    ax.imshow(img)
    ax.set_title(name.upper())
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def span(cov):
    return np.real(np.trace(cov, axis1=-2, axis2=-1))

span_pre = span(results['pre'])
print('Span pre mean/std:', float(span_pre.mean()), float(span_pre.std()))

for key in results:
    if key == 'pre':
        continue
    sp = span(results[key])
    print(f'Span {key} mean/std:', float(sp.mean()), float(sp.std()))

## GRDK Side-by-Side Viewer

Left pane shows pre-filter RGB and right pane shows selected post-filter RGB.

In [ ]:
post_key = 'mulog' if 'mulog' in rgb_results else ('nlsar' if 'nlsar' in rgb_results else 'pre')

viewer = DualGeoViewer()
viewer.set_mode('dual')
viewer.set_array(rgb_results['pre'], pane=0)
viewer.set_array(rgb_results[post_key], pane=1)
viewer.show()

print('Viewer launched with panes: pre vs', post_key)

## Notes

- `CovarianceMatrix` output is CCYX; filtering backends use YXCC.
- RGB mapping in this notebook follows your requested assignment: R=XX, G=XY, B=YY.
- For dual-pol datasets, YY is taken from the last diagonal term.
- NL-SAR requires `nlsartoolbox` and a homogeneous `noise_image`.